## **Random Forest Regressor** ##
The aim for introducing Random Forest (Regressor) to the project is to **predict future number of influenza case detections** with the obtained data. It is chosen because of its low complexity. 

---

In [1]:
# Import libraries needed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv("data_engineed.csv")
df["From"] = pd.to_datetime(df["From"])
df = df.sort_values("From").reset_index(drop=True)

In [3]:
mete_col = ["Diurnal temperature range (°C)", "Mean temperature (°C)", "Mean humidity (%)", "Solar radiation (MJ/m²)", "Air pressure (hPa)", "Dew point temperature (°C)", "Wind speed (km/h)", "Mean UV"]
flu_col = ["H1", "H3", "B", "AandB"]

---

**Grid of hyperparameters**
1. `n_estimators`: The number of tress in the forest
2. `max_features`: The number of features to consider when looking for the best split
    - If float, then `max(1, int(max_features * n_features_in_))` features are considered at each split.
    - If “sqrt”, then `max_features=sqrt(n_features)`
    - If “log2”, then `max_features=log2(n_features)`

3. `max_depth`: The maximum depth of each trees
4. `random_state`: 
5. `min_samples_split`: The minimum number of samples required to split an internal node
6. `min_samples_leaf`: The minimum number of samples required to be at a leaf node

In [4]:
# Hyperparameters grid
grid = {
    "n_estimators": [x for x in range(100, 1000, 100)],
    "max_features": ["sqrt", "log2", 0.8],
    "max_depth": [x for x in range(3, 11)],
    "random_state": [18, 24, 42], 
    "min_samples_split": [2, 5],
    "min_samples_leaf": [5, 10, 20, 30]
}


In [5]:
for col in flu_col: 
    tmp_df = df.copy()

    for n in range(3):
        tmp_df[f"{col}_Lag{n+1}"] = tmp_df[col].shift(n+1)
    tmp_df[f"{col}_volatility"] = tmp_df[col].shift(1).rolling(window=4).std()
    tmp_df[f"{col}_rolling_mean"] = tmp_df[col].shift(1).rolling(window=4).mean()

    tmp_df = tmp_df.dropna().reset_index(drop=True)

    X = tmp_df.drop(columns=["From"]+flu_col)

    print(f"{'='*70} {col} {'='*70}")
    y = tmp_df[col]

    train_end = int(tmp_df.shape[0] * 0.8)
    X_train, X_test = X.iloc[:train_end], X.iloc[train_end:]
    y_train, y_test = y.iloc[:train_end], y.iloc[train_end:]

    y_train_log = np.log1p(y_train)

    tscv = TimeSeriesSplit(n_splits=5)

    rfr = RandomForestRegressor(random_state=42)
    rfr_cv = RandomizedSearchCV(estimator=rfr, param_distributions=grid, cv=tscv, scoring="r2", n_jobs=-1)

    rfr_cv.fit(X_train, y_train_log)
    print(f"Best Parameters: {rfr_cv.best_params_}")

    print(f"\nFeature importances: ")
    model = rfr_cv.best_estimator_
    importance = pd.Series(model.feature_importances_, index=X.columns)
    print(importance.sort_values(ascending=False))

    model.fit(X_train, y_train_log)

    # Evaluation
    y_pred_log = model.predict(X_train)
    y_pred = np.expm1(y_pred_log)
    print(f"\nModel metrics with train data: ")
    print(f"MAE: {mean_absolute_error(y_train, y_pred):.4f}")
    print(f"MSE: {mean_squared_error(y_train, y_pred):.4f}")
    print(f"R2:  {r2_score(y_train, y_pred):.4f}")

    y_pred_log = model.predict(X_test)
    y_pred = np.expm1(y_pred_log)
    print(f"\nModel metrics with test data: ")
    print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
    print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
    print(f"R2:  {r2_score(y_test, y_pred):.4f}")
    

====================================================================== H1 ======================================================================
Best Parameters: {'random_state': 18, 'n_estimators': 800, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': 0.8, 'max_depth': 8}

Feature importances: 
H1_Lag1                           0.719302
H1_rolling_mean                   0.155763
H1_Lag2                           0.081102
H1_volatility                     0.007739
Mean temperature (°C)             0.004778
Week                              0.004650
Solar radiation (MJ/m²)           0.004112
H1_Lag3                           0.003641
Wind speed (km/h)                 0.003471
Dew point temperature (°C)        0.003469
Air pressure (hPa)                0.003326
Diurnal temperature range (°C)    0.003160
Mean humidity (%)                 0.002012
Population                        0.001611
Year                              0.001246
Mean UV                           0.000501
C

---
**Interpretations**
